# 01 — Modelos principais

Reproduz os descritivos da base analítica, as regressões lineares robustas, a medida D e os diagnósticos dos modelos.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan, linear_reset
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import jarque_bera

RAIZ_REPOSITORIO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PASTA_DADOS = RAIZ_REPOSITORIO / "data"
PASTA_RESULTADOS = RAIZ_REPOSITORIO / "outputs"
PASTA_RESULTADOS.mkdir(exist_ok=True)

ORDEM_HIERARQUIAS = ["H1", "H2", "H3", "H4"]
NOME_HIERARQUIA = {
    "H1": "Trânsito rápido",
    "H2": "Arterial",
    "H3": "Coletora",
    "H4": "Local",
}

RENOMEAR_COLUNAS = {
    "H3_R11": "celula_h3_r11",
    "n_min_principal": "minimo_viagens_hierarquia",
    "media_h3_kmh": "velocidade_operacional_media_kmh",
    "PC1_INA": "indice_nivel_atividade_continuo",
    "class_ina_escola": "nivel_atividade",
    "limite_contextual_kmh": "velocidade_contextual_referencia_kmh",
    "delta_v_kmh": "velocidade_insegura_kmh",
    "radar": "fiscalizacao_eletronica",
    "semaforo": "controle_semaforico",
    "controle_cat": "tipo_controle",
    "intersecao": "intersecao_viaria",
    "densidade_classe": "densidade_domiciliar_classe",
    "comercio_relativo": "atividade_economica_relativa",
}

RENOMEAR_CONTROLES = {
    "Nenhum": "Nenhum",
    "Radar apenas": "Fiscalização eletrônica",
    "Semaforo apenas": "Controle semafórico",
    "Radar + semaforo": "Fiscalização e semáforo",
}

## 1. Base analítica

In [ ]:
dados_h3 = pd.read_csv(
    PASTA_DADOS / "base_h3_analitica.csv.gz",
    dtype={"H3_R11": str},
).rename(columns=RENOMEAR_COLUNAS)

dados_h3["tipo_controle"] = dados_h3["tipo_controle"].replace(RENOMEAR_CONTROLES)
dados_h3 = dados_h3.sort_values(["hierarquia", "ordem_espacial"]).reset_index(drop=True)

contagem_h3_esperada = {"H1": 727, "H2": 2140, "H3": 7463, "H4": 8506}
contagem_h3_observada = (
    dados_h3.groupby("hierarquia")["celula_h3_r11"].nunique().to_dict()
)

assert len(dados_h3) == 18_836
assert contagem_h3_observada == contagem_h3_esperada
assert not dados_h3.duplicated(["hierarquia", "celula_h3_r11"]).any()
assert np.allclose(
    dados_h3["velocidade_insegura_kmh"],
    dados_h3["velocidade_operacional_media_kmh"]
    - dados_h3["velocidade_contextual_referencia_kmh"],
)

print(f"Base analítica: {len(dados_h3):,} combinações H3–hierarquia")
pd.Series(contagem_h3_observada, name="n_celulas_h3")

## 2. Descritivos por hierarquia viária

In [ ]:
resumo_por_hierarquia = (
    dados_h3.groupby("hierarquia", sort=False)
    .agg(
        n_h3=("celula_h3_r11", "size"),
        minimo_viagens=("minimo_viagens_hierarquia", "first"),
        viagens_mediana=("n_viagens", "median"),
        viagens_p25=("n_viagens", lambda valores: valores.quantile(0.25)),
        viagens_p75=("n_viagens", lambda valores: valores.quantile(0.75)),
        condutores_mediana=("n_condutores", "median"),
        velocidade_operacional_media=("velocidade_operacional_media_kmh", "mean"),
        velocidade_operacional_dp=("velocidade_operacional_media_kmh", "std"),
        velocidade_operacional_mediana=("velocidade_operacional_media_kmh", "median"),
        velocidade_insegura_media=("velocidade_insegura_kmh", "mean"),
        velocidade_insegura_mediana=("velocidade_insegura_kmh", "median"),
        proporcao_acima_referencia=(
            "velocidade_insegura_kmh",
            lambda valores: (valores > 0).mean(),
        ),
    )
    .reset_index()
)
resumo_por_hierarquia.insert(
    1, "hierarquia_nome", resumo_por_hierarquia["hierarquia"].map(NOME_HIERARQUIA)
)
resumo_por_hierarquia.to_csv(
    PASTA_RESULTADOS / "descritivos_por_hierarquia.csv", index=False
)
print(resumo_por_hierarquia.round(3).to_string(index=False))

## 3. Especificação dos modelos

In [ ]:
def construir_matriz_explicativa(dados_hierarquia, especificacao="completa"):
    if especificacao not in {"completa", "infraestrutura", "contexto"}:
        raise ValueError(especificacao)

    matriz_explicativa = pd.DataFrame(index=dados_hierarquia.index)
    matriz_explicativa["Intercepto"] = 1.0

    if especificacao in {"completa", "infraestrutura"}:
        matriz_explicativa["Fiscalização eletrônica"] = (
            dados_hierarquia["tipo_controle"] == "Fiscalização eletrônica"
        ).astype(float)
        matriz_explicativa["Controle semafórico"] = (
            dados_hierarquia["tipo_controle"] == "Controle semafórico"
        ).astype(float)
        if (dados_hierarquia["tipo_controle"] == "Fiscalização e semáforo").any():
            matriz_explicativa["Fiscalização e semáforo"] = (
                dados_hierarquia["tipo_controle"] == "Fiscalização e semáforo"
            ).astype(float)
        matriz_explicativa["Interseção viária"] = dados_hierarquia[
            "intersecao_viaria"
        ].astype(float)

    if especificacao in {"completa", "contexto"}:
        matriz_explicativa["Densidade domiciliar (média)"] = (
            dados_hierarquia["densidade_domiciliar_classe"] == "Media"
        ).astype(float)
        matriz_explicativa["Densidade domiciliar (alta)"] = (
            dados_hierarquia["densidade_domiciliar_classe"] == "Alta"
        ).astype(float)
        matriz_explicativa["Atividade econômica relativa"] = dados_hierarquia[
            "atividade_economica_relativa"
        ].astype(float)

    return matriz_explicativa


def ajustar_rlm_huber(dados_hierarquia, coluna_resposta, especificacao="completa"):
    matriz_explicativa = construir_matriz_explicativa(dados_hierarquia, especificacao)
    variavel_resposta = dados_hierarquia[coluna_resposta].astype(float)
    modelo_rlm = sm.RLM(
        variavel_resposta,
        matriz_explicativa,
        M=sm.robust.norms.HuberT(t=1.345),
    ).fit(cov="H1", maxiter=200)
    return modelo_rlm, matriz_explicativa, variavel_resposta


def calcular_medida_d(valores_observados, valores_ajustados):
    valores_observados = np.asarray(valores_observados, dtype=float)
    valores_ajustados = np.asarray(valores_ajustados, dtype=float)
    soma_quadrados_residuos = np.sum((valores_observados - valores_ajustados) ** 2)
    soma_quadrados_total = np.sum(
        (valores_observados - valores_observados.mean()) ** 2
    )
    return 1 - soma_quadrados_residuos / soma_quadrados_total

## 4. Regressões lineares robustas e medida D

In [ ]:
registros_coeficientes_rlm = []
registros_residuos_rlm = []
registros_medida_d = []

for hierarquia in ORDEM_HIERARQUIAS:
    dados_hierarquia = dados_h3.loc[dados_h3["hierarquia"] == hierarquia].copy()

    for nome_resposta, coluna_resposta, especificacao in [
        ("velocidade_operacional_media", "velocidade_operacional_media_kmh", "completa"),
        ("velocidade_insegura", "velocidade_insegura_kmh", "completa"),
        ("velocidade_insegura_infraestrutura", "velocidade_insegura_kmh", "infraestrutura"),
    ]:
        modelo_rlm, matriz_explicativa, variavel_resposta = ajustar_rlm_huber(
            dados_hierarquia, coluna_resposta, especificacao
        )

        for variavel_explicativa in matriz_explicativa.columns:
            coeficiente = float(modelo_rlm.params[variavel_explicativa])
            erro_padrao = float(modelo_rlm.bse[variavel_explicativa])
            registros_coeficientes_rlm.append({
                "hierarquia": hierarquia,
                "hierarquia_nome": NOME_HIERARQUIA[hierarquia],
                "resposta": nome_resposta,
                "variavel": variavel_explicativa,
                "beta": coeficiente,
                "ep_H1": erro_padrao,
                "p": float(modelo_rlm.pvalues[variavel_explicativa]),
                "ic95_inf": coeficiente - 1.96 * erro_padrao,
                "ic95_sup": coeficiente + 1.96 * erro_padrao,
            })

        if nome_resposta == "velocidade_operacional_media":
            registros_residuos_rlm.extend(pd.DataFrame({
                "hierarquia": hierarquia,
                "celula_h3_r11": dados_hierarquia["celula_h3_r11"].to_numpy(),
                "velocidade_operacional_media_predita_kmh": np.asarray(
                    modelo_rlm.predict(matriz_explicativa)
                ),
                "residuo_velocidade_operacional_media_kmh": np.asarray(modelo_rlm.resid),
            }).to_dict("records"))

    for bloco_variaveis, especificacao in [
        ("Infraestrutura e operação", "infraestrutura"),
        ("Contexto urbano", "contexto"),
        ("Modelo completo", "completa"),
    ]:
        modelo_rlm, matriz_explicativa, variavel_resposta = ajustar_rlm_huber(
            dados_hierarquia, "velocidade_operacional_media_kmh", especificacao
        )
        registros_medida_d.append({
            "hierarquia": hierarquia,
            "hierarquia_nome": NOME_HIERARQUIA[hierarquia],
            "bloco": bloco_variaveis,
            "D": calcular_medida_d(
                variavel_resposta, modelo_rlm.predict(matriz_explicativa)
            ),
        })

coeficientes_rlm = pd.DataFrame(registros_coeficientes_rlm)
residuos_rlm = pd.DataFrame(registros_residuos_rlm)
medidas_d = pd.DataFrame(registros_medida_d)

coeficientes_rlm.to_csv(PASTA_RESULTADOS / "coeficientes_RLM.csv", index=False)
residuos_rlm.to_csv(
    PASTA_RESULTADOS / "residuos_RLM.csv.gz", index=False, compression="gzip"
)
medidas_d.to_csv(PASTA_RESULTADOS / "medida_D.csv", index=False)

print(
    coeficientes_rlm.query("resposta == 'velocidade_operacional_media'")
    .round(4)
    .to_string(index=False)
)
print(medidas_d.round(4).to_string(index=False))

## 5. Diagnósticos dos modelos lineares

In [ ]:
registros_diagnosticos_mqo = []
registros_vif = []

for hierarquia in ORDEM_HIERARQUIAS:
    dados_hierarquia = dados_h3.loc[dados_h3["hierarquia"] == hierarquia].copy()
    matriz_explicativa = construir_matriz_explicativa(dados_hierarquia, "completa")

    for indice_variavel, variavel_explicativa in enumerate(matriz_explicativa.columns):
        if variavel_explicativa != "Intercepto":
            registros_vif.append({
                "hierarquia": hierarquia,
                "variavel": variavel_explicativa,
                "VIF": float(variance_inflation_factor(
                    matriz_explicativa.to_numpy(float), indice_variavel
                )),
            })

    for nome_resposta, coluna_resposta in [
        ("velocidade_operacional_media", "velocidade_operacional_media_kmh"),
        ("velocidade_insegura", "velocidade_insegura_kmh"),
    ]:
        variavel_resposta = dados_hierarquia[coluna_resposta].astype(float)
        modelo_mqo = sm.OLS(variavel_resposta, matriz_explicativa).fit()
        residuos = np.asarray(modelo_mqo.resid, dtype=float)

        estatistica_jb, p_jb, assimetria, curtose = jarque_bera(residuos)
        resultado_bp = het_breuschpagan(residuos, matriz_explicativa)
        influencia = modelo_mqo.get_influence()
        distancia_cook = np.asarray(influencia.cooks_distance[0], dtype=float)
        residuos_studentizados = np.asarray(
            influencia.resid_studentized_internal, dtype=float
        )

        try:
            resultado_reset = linear_reset(modelo_mqo, power=2, use_f=True)
            estatistica_reset = float(resultado_reset.fvalue)
            p_reset = float(resultado_reset.pvalue)
        except Exception:
            estatistica_reset = np.nan
            p_reset = np.nan

        n_observacoes = len(dados_hierarquia)
        registros_diagnosticos_mqo.append({
            "hierarquia": hierarquia,
            "hierarquia_nome": NOME_HIERARQUIA[hierarquia],
            "resposta": nome_resposta,
            "n": n_observacoes,
            "jarque_bera": estatistica_jb,
            "jb_p": p_jb,
            "assimetria": assimetria,
            "curtose": curtose,
            "breusch_pagan_lm": resultado_bp[0],
            "bp_p": resultado_bp[1],
            "reset_f": estatistica_reset,
            "reset_p": p_reset,
            "prop_cook_gt_4_n": np.mean(distancia_cook > 4 / n_observacoes),
            "prop_abs_resid_student_gt_3": np.mean(
                np.abs(residuos_studentizados) > 3
            ),
            "max_cook": distancia_cook.max(),
        })

diagnosticos_mqo = pd.DataFrame(registros_diagnosticos_mqo)
valores_vif = pd.DataFrame(registros_vif)

diagnosticos_mqo.to_csv(PASTA_RESULTADOS / "diagnosticos_MQO.csv", index=False)
valores_vif.to_csv(PASTA_RESULTADOS / "VIF.csv", index=False)

print(diagnosticos_mqo.round(4).to_string(index=False))
print(
    valores_vif.pivot(index="variavel", columns="hierarquia", values="VIF")
    .round(2)
    .to_string()
)

## Saídas

Os arquivos deste notebook são gravados em `outputs/` e utilizados pelo notebook 02.